# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (Scoring/Ranking)

I am framing Lane A as a Scoring and Ranking task. We want to assign a numerical continuous score to each page representing its click-performance deficit (the gap between expected and actual performance). We then rank these pages in descending order so the client can address the worst-performing culprits first. While it could be modeled as a binary classification problem (e.g., "Underperforming" vs. "Healthy"), ranking is vastly superior because it naturally constructs a prioritized workflow matching the client's limited optimization budget.

In [10]:
# ML Task Type confirmed: Continuous Scoring and Ranking approach.

## 2. Target or proxy

The target variable is the **CTR Performance Gap**, which acts as a proxy for optimization potential. The target is defined mathematically as:
Target = Expected_CTR - Observed_CTR
The Observed CTR comes straight from historical Google Search Console observations. The Expected CTR is a dynamically generated label representing a baseline benchmark value for pages holding that identical organic rank tier across similar search environments.

In [11]:
# Target definition mapping complete.

## 3. Success metric

**Business Success Metric:** Total click-yield capture (measuring net organic traffic gains over a 30-day post-optimization validation window).
**ML Evaluation Metric:** Mean Absolute Error (MAE) calculated on our performance gap scores, backed by Precision at K (P@K) where K=50. Since copywriters have limited hours to refresh the top 50 flagged pages, we must guarantee that the pages placed at the top of the queue are genuinely underperforming. A low MAE ensures our model baseline is calibrated accurately.
What to put in the code cell directly below Section 3:

In [12]:
# Evaluation framework: MAE and Precision@K targeted.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
**Unit of Analysis:** One row = One unique page URL path.

We slice our dataset to capture structural performance features bound to individual asset paths. This ensures the output can map cleanly onto discrete internal marketing execution tasks (e.g., flagging specific URLs for a content rewrite).

In [13]:
import pandas as pd
import numpy as np

# 1. Download dataset directly from upstream repository source
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Clean up column names to lowercase/stripped whitespace just in case
df.columns = df.columns.str.strip().str.lower()

# Map potential column variations safely
rename_dict = {}
for col in df.columns:
    if 'url' in col or 'path' in col:
        rename_dict[col] = 'page_path'
    elif 'impressions' in col:
        rename_dict[col] = 'impressions_90d'
    elif 'clicks' in col:
        rename_dict[col] = 'clicks_90d'
    elif 'position' in col:
        rename_dict[col] = 'position_avg'
    elif 'ctr' in col:
        rename_dict[col] = 'ctr'

df = df.rename(columns=rename_dict)

# Handle duplicate column names that might arise from the rename operation.
# If multiple original columns mapped to the same new name (e.g., 'avg_position' and 'position_tier' both map to 'position_avg'),
# Pandas will create a DataFrame with multiple columns having that same name.
# To ensure `df['column_name']` selects a Series, we need to ensure unique column names.
# We'll keep the first occurrence of any duplicate column name and drop the rest.
df = df.loc[:, ~df.columns.duplicated(keep='first')]

# 2. Extract key features defining our unit of analysis
keep_cols = [c for c in ['page_path', 'impressions_90d', 'clicks_90d', 'position_avg', 'ctr'] if c in df.columns]
unit_of_analysis = df[keep_cols].copy()

# Fix Type Errors: Force features to numeric types, converting invalid entries to NaN safely
if 'position_avg' in unit_of_analysis.columns:
    unit_of_analysis['position_avg'] = pd.to_numeric(unit_of_analysis['position_avg'], errors='coerce')
if 'ctr' in unit_of_analysis.columns:
    unit_of_analysis['ctr'] = pd.to_numeric(unit_of_analysis['ctr'], errors='coerce')

# Drop any rows where core metrics are missing or parsed incorrectly
unit_of_analysis = unit_of_analysis.dropna(subset=['position_avg', 'ctr'])

# 3. Sketch a mock target column (CTR performance gap relative to a baseline curve)
expected_ctr = 0.12 / (unit_of_analysis['position_avg'] + 0.5)
unit_of_analysis['ctr_gap_target'] = (expected_ctr - unit_of_analysis['ctr']).clip(lower=0)

print("--- UNIT OF ANALYSIS DATAFRAME ---")
print(f"One row represents one unique page path. Total rows: {len(unit_of_analysis):,}\n")
print(unit_of_analysis.head(5))

--- UNIT OF ANALYSIS DATAFRAME ---
One row represents one unique page path. Total rows: 30,000

   impressions_90d  clicks_90d  position_avg   ctr  ctr_gap_target
0             3803          29          10.6  0.76             0.0
1            15320           7          20.3  0.05             0.0
2            12581          11          36.5  0.09             0.0
3            11751          58           6.2  0.49             0.0
4            19140          24          44.0  0.13             0.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A static heuristic ruleset—such as flagging everything with a CTR under 2%—fails because organic click expectations scale exponentially with average rank positions. A 1.8% CTR is catastrophic for a page sitting at rank #1, but represents an elite, highly converting page if it's sitting deep at rank #9. Because real click distributions scale non-linearly across structural layout variables, impressions, and device types, an hardcoded threshold will flood the execution team with false positives. An ML model builds a dynamic baseline curve that automatically handles multiple inputs simultaneously.
Click Close on the markdown toolbar to save it.

In [14]:
# Rule limitations analyzed: ML dynamically scales across continuous positions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.